In [1]:
import json
import os
import sys
import pandas as pd
from pprint import pprint
from tqdm import tqdm
from langchain import PromptTemplate, FewShotPromptTemplate

sys.path.insert(0, '/home/ameyh/mental-health-comorbitidy-classification/src/')
from prompts import DEPRESSION_FEWSHOT_LANGCHAIN

tqdm.pandas()

In [2]:
DEPRESSION_FEWSHOT_LANGCHAIN

{'few_shot_prefix': "\nBelow are posts and their respective assessments based on the criteria for clinical depression as defined in the DSM-5.\nFormat your response as a JSON object /{'depression':''/} with values either 'yes' or 'no'.\n",
 'prompt_template': '\nPost: {post}\nAssesement: {label}\n',
 'few_shot_suffix': '\nBased on the above, assess the content of the following post:\nPost: {post}\nAssessment:\n'}

In [38]:
test_path = '/home/ameyh/mental-health-comorbitidy-classification/data/test/full_test.csv'
corpus_path = '/home/ameyh/mental-health-comorbitidy-classification/data/silver_data/silver_labels_gpt.csv'
mapping_path_depression = "/home/ameyh/mental-health-comorbitidy-classification/data/mappings/semantic-similarity-depression.json"
mapping_path_anxiety = "/home/ameyh/mental-health-comorbitidy-classification/data/mappings/semantic-similarity-anxiety.json"
mapping_path_comorbid = "/home/ameyh/mental-health-comorbitidy-classification/data/mappings/semantic-similarity-comorbid.json"
mapping_path_normal = "/home/ameyh/mental-health-comorbitidy-classification/data/mappings/semantic-similarity-normal.json"


df_test = pd.read_csv(test_path).reset_index()
df_corpus = pd.read_csv(corpus_path).reset_index()
mapping_depression = json.load(open(mapping_path_depression))
mapping_anxiety = json.load(open(mapping_path_anxiety))
mapping_comorbid = json.load(open(mapping_path_comorbid))
mapping_normal = json.load(open(mapping_path_normal))


print('-'*50)
print(f"Test set size: {df_test.shape[0]}")
print(f"Corpus (Silver Labels) size: {df_corpus.shape[0]}")
print(f"Total mapping (Depression): {len(mapping_depression)}")
print(f"Total mapping (Anxiety): {len(mapping_anxiety)}")
print(f"Total mapping (Comorbid): {len(mapping_comorbid)}")
print(f"Total mapping (Normal): {len(mapping_normal)}")
print('-'*50)


--------------------------------------------------
Test set size: 2872
Corpus (Silver Labels) size: 7667
Total mapping (Depression): 2872
Total mapping (Anxiety): 2872
Total mapping (Comorbid): 2872
Total mapping (Normal): 2872
--------------------------------------------------


In [39]:
def merge_dicts(dict1, dict2):
    merged_dict = {}
    for i in dict1.keys():
        merged_dict[i] = dict1[i]
    for i in dict2.keys():
        merged_dict[i] = dict2[i]
    return merged_dict

df_corpus['depression_label'] = df_corpus['depression_label'].apply(lambda label: {"depression": "yes"} if label == 1 else {"depression": "no"})
df_corpus['anxiety_label'] = df_corpus['anxiety_label'].apply(lambda label: {"anxiety": "yes"} if label == 1 else {"anxiety": "no"})
df_corpus['comorbidity_label'] = df_corpus.apply(lambda row: merge_dicts(row['depression_label'],row['anxiety_label']), axis=1)
df_corpus['comorbidity_label'].value_counts()

{'depression': 'no', 'anxiety': 'no'}      3349
{'depression': 'yes', 'anxiety': 'no'}     2705
{'depression': 'no', 'anxiety': 'yes'}     1048
{'depression': 'yes', 'anxiety': 'yes'}     565
Name: comorbidity_label, dtype: int64

In [40]:
df_corpus.head(1)

,level_0,index,text,subreddit,author,system_description,prompt,gpt_prediction,Mental Health Disorder,Name of Mental Health Disorder,DSM5 Rationale,silver_label,id,depression_label,anxiety_label,multilabel_clf_label,multiclass_clf_label,comorbidity_label
0,0,11,"i woke up very early, 2 am. i just came out of...",ForeverAlone,SixViking,You are a Psychology professor working in the ...,"Reddit Post: ""i woke up very early, 2 am. i ju...",Mental Health Disorder: Yes\nName of Mental He...,Yes,Major Depressive Disorder (MDD),The language used in the post indicates severa...,Depression,4JEVyZ,{'depression': 'yes'},{'anxiety': 'no'},"[1, 0]",Depression,"{'depression': 'yes', 'anxiety': 'no'}"


In [64]:
DEPRESSION_FEWSHOT_LANGCHAIN = {
"few_shot_prefix": '''
Below are posts and their respective assessments based on the criteria for clinical depression as defined in the DSM-5.
Format your response as a JSON object {'depression': ''} with values either 'yes' or 'no'.
'''
,
"prompt_template": lambda post, label: f'''
Post: {post}
Assesement: {label}
'''
,
"few_shot_suffix": lambda post: f'''
Based on the above, assess the content of the following post:
Post: {post}
Assessment:
'''
}

ANXIETY_FEWSHOT_LANGCHAIN = {
"few_shot_prefix": '''
Below are posts and their respective assessments based on the criteria for clinical anxiety as defined in the DSM-5.
Format your response as a JSON object {'anxiety':''} with values either 'yes' or 'no'.
'''
,
"prompt_template": lambda post, label: f'''
Post: {post}
Assesement: {label}
'''
,
"few_shot_suffix": lambda post: f'''
Based on the above, assess the content of the following post:
Post: {post}
Assessment:
'''
}

COMORBIDITY_FEWSHOT_LANGCHAIN = {
"few_shot_prefix": '''
Below are posts and their respective assessments based on the criteria for clinical depression and clinical anxiety respectively as defined in the DSM-5.
Format your response as a JSON object {'depression': '', 'anxiety': ''} with values either 'yes' or 'no'.
'''
,
"prompt_template": lambda post, label: f'''
Post: {post}
Assesement: {label}
'''
,
"few_shot_suffix": lambda post: f'''
Based on the above, assess the content of the following post:
Post: {post}
Assessment:
'''
}

In [67]:
# Generate Few Shot prompts for each data point in test set.
topk = 4 # Provide only even number of examples, 2, 4, 6, 8, ..
text_col = 'text'
id_col = 'id'
task_type = 'comorbidity' #(depression | anxiety | comorbidity)

prompts = []
for i in tqdm(range(df_test.shape[0]), desc=f"Generating few shot prompts"):
    few_shot_examples = []
    input_post = df_test.iloc[i][text_col]
    
    if task_type == 'depression':        
        exemplars_d = mapping_depression[df_test.iloc[i][id_col]]
        exemplar_ids_d = [i[0] for i in exemplars_d][:int(topk/2)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/2)]
        exemplar_ids = exemplar_ids_d + exemplar_ids_n
 
        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])
        
        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['depression_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = DEPRESSION_FEWSHOT_LANGCHAIN['prompt_template']
        few_shot_prefix = DEPRESSION_FEWSHOT_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = DEPRESSION_FEWSHOT_LANGCHAIN['few_shot_suffix'](input_post)
        
    
    elif task_type == 'anxiety':
        exemplars_a = mapping_anxiety[df_test.iloc[i][id_col]]
        exemplar_ids_a = [i[0] for i in exemplars_a][:int(topk/2)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/2)]
        exemplar_ids = exemplar_ids_a + exemplar_ids_n

        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])

        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['anxiety_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = ANXIETY_FEWSHOT_LANGCHAIN['prompt_template']
        few_shot_prefix = ANXIETY_FEWSHOT_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = ANXIETY_FEWSHOT_LANGCHAIN['few_shot_suffix'](input_post)
        
        
    elif task_type == 'comorbidity':
        if topk < 4: 
            topk = 4

        exemplars_d = mapping_depression[df_test.iloc[i][id_col]]
        exemplar_ids_d = [i[0] for i in exemplars_d][:int(topk/4)]
        exemplars_a = mapping_anxiety[df_test.iloc[i][id_col]]
        exemplar_ids_a = [i[0] for i in exemplars_a][:int(topk/4)]
        exemplars_c = mapping_comorbid[df_test.iloc[i][id_col]]
        exemplar_ids_c = [i[0] for i in exemplars_c][:int(topk/4)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/4)]
        exemplar_ids = exemplar_ids_d + exemplar_ids_a + exemplar_ids_c + exemplar_ids_n

        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])

        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['comorbidity_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = COMORBIDITY_FEWSHOT_LANGCHAIN['prompt_template']
        few_shot_prefix = COMORBIDITY_FEWSHOT_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = COMORBIDITY_FEWSHOT_LANGCHAIN['few_shot_suffix'](input_post)
               
        
    few_shot_examples = ''.join(prompt_template(x['post'], x['label']) for x in few_shot_examples)
    few_shot_prompt = ''.join([few_shot_prefix, few_shot_examples, few_shot_suffix])        
    prompts.append(few_shot_prompt)


df_test[f'few_shot_prompt_{task_type}'] = prompts
df_test[f'few_shot_prompt_{task_type}']

Generating few shot prompts:   0%|          | 0/2872 [00:00<?, ?it/s]

Generating few shot prompts: 100%|██████████| 2872/2872 [00:04<00:00, 613.91it/s]


0       \nBelow are posts and their respective assessm...
1       \nBelow are posts and their respective assessm...
2       \nBelow are posts and their respective assessm...
3       \nBelow are posts and their respective assessm...
4       \nBelow are posts and their respective assessm...
                              ...                        
2867    \nBelow are posts and their respective assessm...
2868    \nBelow are posts and their respective assessm...
2869    \nBelow are posts and their respective assessm...
2870    \nBelow are posts and their respective assessm...
2871    \nBelow are posts and their respective assessm...
Name: few_shot_prompt_comorbidity, Length: 2872, dtype: object